[pretrain](https://github.com/OpenGVLab/VideoMAEv2/blob/master/run_mae_pretraining.py)

In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')

In [2]:
from pathlib import Path
import math
import time
import random
import datetime
from functools import partial

import torch
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2
    
from computer_vision.video_mae.parameter_parser import parser
from computer_vision.video_mae.dataset.pretrained_datasets import HybridVideoMAE, DataAugmentationForVideoMAEv2
from computer_vision.video_mae.models.modeling_pretrain import PretrainVisionTransformer, pretrain_videomae_tiny_patch16_224
from computer_vision.video_mae.utils import seed_worker, multiple_pretrain_samples_collate, cosine_scheduler, get_grad_norm
from computer_vision.video_mae.optim_factory import create_optimizer

In [3]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'

mini_train=False
if not mini_train:
    output_dirpath=Path('D:/results/ucf101/video_mae/train') 
    arguments= f"""--data_root {root} --data_path {annotation_path} --output_dir {output_dirpath} 
    --mask_type tube --mask_ratio 0.9 --decoder_mask_type run_cell --decoder_mask_ratio 0.5
    --decoder_depth 4 --with_checkpoint --cos_attn --num_frame 16 --sampling_rate 4 --num_sample 1 
    --opt adamw --lr 6e-4 --clip_grad 0.02 --opt_betas 0.9 0.95 --warmup_epochs 30 
    --batch-size 24 --num_workers 0 --print_freq 30 --time 13
    --device cuda --epochs 300 --resume
    """ # --use-cutmix-mixup
else:
    output_dirpath=Path('D:/results/ucf101/video_mae/mini_train') 
    arguments= f"""--data_root {root} --data_path {annotation_path}  --output_dir {output_dirpath}  
    --mask_type tube --mask_ratio 0.9 --decoder_mask_type run_cell --decoder_mask_ratio 0.5
    --decoder_depth 4 --with_checkpoint --cos_attn --num_frame 16 --sampling_rate 4 --num_sample 1 
    --opt adamw --lr 6e-4 --clip_grad 0.02 --opt_betas 0.9 0.95 --warmup_epochs 30 
    --batch-size 24 --num_workers 0 --print_freq 20  --plot_freq 3
    --epochs 300 --device cuda --n_steps 60  --n_epochs 6 --time 0.5 --resume
    """ # --use-cutmix-mixup --time 18 --resume
args=parser.parse_args(arguments.split())


args.output_dir=Path(args.output_dir)
args.output_dir.mkdir(parents=True, exist_ok=True)
args.checkpoint_dir=args.output_dir/"checkpoints"
args.checkpoint_dir.mkdir(parents=True, exist_ok=True)
args.last=args.checkpoint_dir/args.last
args.best=args.checkpoint_dir/args.best
print(f"{args.last=}")
print(f"{args.best=}")

args.last=WindowsPath('D:/results/ucf101/video_mae/train/checkpoints/last.pth')
args.best=WindowsPath('D:/results/ucf101/video_mae/train/checkpoints/best.pth')


In [4]:
device=torch.device(args.device) if (torch.cuda.is_available() and args.device=='cuda') else torch.device('cpu')
print(f"{device=}")

torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

model=pretrain_videomae_tiny_patch16_224(args)
patch_size=model.encoder.patch_embed.patch_size
print(f"Patch size {patch_size}")
args.window_size=(args.num_frames//args.tubelet_size, args.input_size//patch_size[0], args.input_size//patch_size[1]) # T,H,W
args.patch_size=patch_size

# build dataset
transform = DataAugmentationForVideoMAEv2(args)
train_dataset=HybridVideoMAE(root=args.data_root, setting=args.data_path, train=True, test_mode=False, name_pattern=args.fname_tmpl, 
                       video_ext='avi', is_color=True, modality='rgb', num_segments=1, num_crop=1, new_length=args.num_frames, 
                       new_step=args.sampling_rate, transform=transform, temporal_jitter=False, lazy_init=False, num_sample=args.num_sample)

num_training_steps_per_epoch=len(train_dataset)//args.batch_size
print(f"{num_training_steps_per_epoch=}")


collate_func=None
if args.num_sample>1: collate_func=partial(multiple_pretrain_samples_collate, fold=False)
num_devices=torch.cuda.device_count() # number of CUDA devices
# MUST KEEP args.num_workers to ZERO
train_dataloader=torch.utils.data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, 
                                             pin_memory=num_devices>0 and args.pin_mem, drop_last=len(train_dataset)%args.batch_size!=0, 
                                             worker_init_fn=seed_worker, persistent_workers=args.num_workers>0, collate_fn=collate_func)
data=next(iter(train_dataloader))
process_video, encoder_mask, decoder_mask=data
print(f"{type(process_video)=}, {process_video.shape=}, {process_video.dtype=}, ({process_video.min().item():.3f},{process_video.max().item():.3f})")
print(f"{type(encoder_mask)=}, {encoder_mask.shape=}, {encoder_mask.dtype=}, ({encoder_mask.min().item():.3f},{encoder_mask.max().item():.3f})")
print(f"{type(decoder_mask)=}, {decoder_mask.shape=}, {decoder_mask.dtype=}, ({decoder_mask.min().item():.3f},{decoder_mask.max().item():.3f})")
print(f"{args.num_sample*args.batch_size=}")

device=device(type='cuda')
Patch size (16, 16)
num_training_steps_per_epoch=397
type(process_video)=<class 'torch.Tensor'>, process_video.shape=torch.Size([24, 3, 16, 224, 224]), process_video.dtype=torch.float32, (-2.118,2.640)
type(encoder_mask)=<class 'torch.Tensor'>, encoder_mask.shape=torch.Size([24, 8, 196]), encoder_mask.dtype=torch.uint8, (0.000,1.000)
type(decoder_mask)=<class 'torch.Tensor'>, decoder_mask.shape=torch.Size([24, 8, 196]), decoder_mask.dtype=torch.uint8, (0.000,1.000)
args.num_sample*args.batch_size=24


In [5]:
checkpoint=None
if args.resume and args.last.is_file():
    checkpoint=torch.load(args.last, map_location='cpu', weights_only=False)
    print(f"Resume from checkpoint: {args.last}")
    model.load_state_dict(checkpoint['model'])

model.to(device)
n_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters: {n_parameters/1e6} million")
print(f"Number of training steps per epoch {num_training_steps_per_epoch}")
print(f"Number of training examples per epoch {args.batch_size*num_training_steps_per_epoch}")

optimizer=create_optimizer(args, model)
print("Use step level LR & WD scheduler!")
lr_schedule_values=cosine_scheduler(args.lr, args.min_lr, args.epochs, num_training_steps_per_epoch, warmup_epochs=args.warmup_epochs,
                                   warmup_steps=args.warmup_steps)

if args.weight_decay_end is None: args.weight_decay_end=args.weight_decay
wd_schedule_values=cosine_scheduler(args.weight_decay, args.weight_decay_end, args.epochs, num_training_steps_per_epoch)
print(f"Max WD={wd_schedule_values.max():.7f}, Min WD={wd_schedule_values.min():.7f}")

best_loss=float('inf')
if checkpoint is not None:
    optimizer.load_state_dict(checkpoint['optimizer'])
    args.start_epoch=checkpoint['epoch']+1
    best_loss=checkpoint['best_loss']

torch.cuda.empty_cache()
print(f"Start training for {args.epochs} epochs at {args.start_epoch}")

Number of parameters: 5.634284 million
Number of training steps per epoch 397
Number of training examples per epoch 9528
Param group {
  "no_decay": {
    "weight_decay": 0.0,
    "params": [
      "mask_token",
      "encoder.patch_embed.proj.bias",
      "encoder.blocks.0.gamma_1",
      "encoder.blocks.0.gamma_2",
      "encoder.blocks.0.norm1.weight",
      "encoder.blocks.0.norm1.bias",
      "encoder.blocks.0.attn.scale",
      "encoder.blocks.0.attn.q_bias",
      "encoder.blocks.0.attn.v_bias",
      "encoder.blocks.0.attn.proj.bias",
      "encoder.blocks.0.norm2.weight",
      "encoder.blocks.0.norm2.bias",
      "encoder.blocks.0.mlp.fc1.bias",
      "encoder.blocks.0.mlp.fc2.bias",
      "encoder.blocks.1.gamma_1",
      "encoder.blocks.1.gamma_2",
      "encoder.blocks.1.norm1.weight",
      "encoder.blocks.1.norm1.bias",
      "encoder.blocks.1.attn.scale",
      "encoder.blocks.1.attn.q_bias",
      "encoder.blocks.1.attn.v_bias",
      "encoder.blocks.1.attn.proj.bias",

In [6]:
from computer_vision.video_mae.utils import MetricLogger, save_checkpoint, form_stats
from computer_vision.video_mae.engine_for_pretraining import train_one_epoch
from computer_vision.torch_video.utils.plotting import plot_all
from computer_vision.torch_video.utils.progress import save_metrics

In [7]:
start_time=time.time()
stop=False
for epoch in range(args.start_epoch, args.epochs):
    if args.n_epochs is not None and epoch>args.n_epochs-1:
        print(f"Hit desired number of epochs {epoch}/{args.n_epochs}--break")
        break
    start_epoch_time=time.time()
    train_metric_logger=MetricLogger(delimiter=" ")
    train_stats=train_one_epoch(model, train_dataloader, optimizer, device, epoch, max_norm=args.clip_grad,
                                tubelet_size=args.tubelet_size, patch_size=args.patch_size, normalize_target=args.normalize_target, 
                                lr_scheduler=None, start_steps=epoch*num_training_steps_per_epoch, lr_schedule_values=lr_schedule_values, 
                                wd_schedule_values=wd_schedule_values, print_freq=args.print_freq, metric_logger=train_metric_logger,
                                n_steps=args.n_steps)

    if args.output_dir is not None:
        save_checkpoint(args.last, model, optimizer, epoch, best_loss, scaler=None)
        stats=form_stats(train_metric_logger, mode='train')#|form_stats(val_metric_logger, mode='val')
        save_metrics(args.output_dir/"result.csv", stats, epoch=epoch, start_epoch_time=start_epoch_time)
        if args.plot_freq>0 and (epoch+1)%args.plot_freq==0: plot_all(args.output_dir/"result.csv")

    if best_loss>train_metric_logger.meters['loss'].global_avg:
        best_loss=train_metric_logger.meters['loss'].global_avg
        save_checkpoint(args.best, model, optimizer, epoch, best_loss, scaler=None)
    
    if args.time is not None:
        stop|=(time.time()-start_time)>(args.time*3600)
    if stop: break
            
total_time=time.time()-start_time
total_time_str=str(datetime.timedelta(seconds=int(total_time)))
print(f"Training time {total_time_str}")

Epoch: [0] [  0/397] eta:0:09:58 lr:0.000000 min_lr:0.000000 loss:1.1383 (1.1383) weight_decay:0.0500 (0.0500) grad_norm:0.0718 (0.0718) time: 1.5072 (1.5072 -- 1.5072) data: 0.9186 (0.9186 -- 0.9186) max mem: 1473
Epoch: [0] [ 30/397] eta:0:06:59 lr:0.000002 min_lr:0.000002 loss:1.1202 (1.1226) weight_decay:0.0500 (0.0500) grad_norm:0.0720 (0.0720) time: 1.1186 (1.0563 -- 1.2345) data: 0.9223 (0.8579 -- 1.0293) max mem: 1518
Epoch: [0] [ 60/397] eta:0:06:28 lr:0.000003 min_lr:0.000003 loss:1.1172 (1.1200) weight_decay:0.0500 (0.0500) grad_norm:0.0715 (0.0718) time: 1.1827 (1.0885 -- 2.1547) data: 0.9744 (0.8924 -- 1.8113) max mem: 1518
Epoch: [0] [ 90/397] eta:0:05:51 lr:0.000005 min_lr:0.000005 loss:1.1196 (1.1195) weight_decay:0.0500 (0.0500) grad_norm:0.0705 (0.0714) time: 1.1198 (1.0476 -- 1.1790) data: 0.9247 (0.8636 -- 0.9832) max mem: 1518
Epoch: [0] [120/397] eta:0:05:19 lr:0.000006 min_lr:0.000006 loss:1.1156 (1.1184) weight_decay:0.0500 (0.0500) grad_norm:0.0689 (0.0709) tim